In [ ]:
!pip install roboflow ultralytics

In [ ]:
# Google Driveをマウント（Colab環境の場合）
import sys
import os
from pathlib import Path

try:
    import google.colab
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/Visuable_for_you_tabletennis"
    os.chdir(PROJECT_ROOT)
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "scripts/notebooks"))
    print(f"✓ Google Driveをマウントしました")
    print(f"✓ プロジェクトルート: {PROJECT_ROOT}")
    IN_COLAB = True

except ImportError:
    IN_COLAB = False
    notebook_dir = Path(__file__).parent if "__file__" in globals() else Path.cwd()
    sys.path.insert(0, str(notebook_dir))
    print(f"✓ ローカル環境で実行中")
    print(f"✓ 作業ディレクトリ: {Path.cwd()}")

# utilsをインポート
from utils import ColabFileManager, ConfigLoader

# ファイルマネージャーの初期化（プロジェクトルートは自動検出）
fm = ColabFileManager()

print(f"✓ ファイルマネージャー初期化完了")
print(f"  - 検出されたプロジェクトルート: {fm.project_root}")
print(f"  - Colab環境: {fm.is_colab}")

In [ ]:
from roboflow import Roboflow
import json
import os
from pathlib import Path

# api.config
if fm.is_colab:
    config_path = "configs/api.config"

    if config_path:
        config = ConfigLoader.load_json(config_path)
    else:
        raise FileNotFoundError("設定ファイルがアップロードされませんでした")
else:
    # ローカル環境の場合
    config = ConfigLoader.load_json(fm.get_path("configs/api.config"))

roboflow_config = config["roboflow"]

rf = Roboflow(api_key=roboflow_config["api_key"])
project = rf.workspace(roboflow_config["workspace"]).project(roboflow_config["project"])
version = project.version(roboflow_config["version"])

dataset = version.download(model_format="yolov11", location="/content/datasets")

print(f"データセットのダウンロード完了: {dataset.location}")
print("注意: データセットは/content（一時）に保存されています")

In [ ]:
import os
import yaml

data_yaml_path = os.path.join(dataset.location, "data.yaml")
with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

print("データセット設定:")
print(yaml.dump(data_config, default_flow_style=False, allow_unicode=True))

train_images = os.path.join(dataset.location, "train", "images")
valid_images = os.path.join(dataset.location, "valid", "images")

train_count = len(
    [f for f in os.listdir(train_images) if f.endswith((".jpg", ".png", ".jpeg"))]
)
valid_count = len(
    [f for f in os.listdir(valid_images) if f.endswith((".jpg", ".png", ".jpeg"))]
)

print(f"\n訓練データ画像数: {train_count}")
print(f"検証データ画像数: {valid_count}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path


def visualize_sample_with_labels(image_path, label_path, class_names):
    """画像とアノテーションを可視化"""
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            labels = f.readlines()

        for label in labels:
            parts = label.strip().split()
            class_id = int(parts[0])
            x_center, y_center, width, height = map(float, parts[1:5])

            x1 = int((x_center - width / 2) * w)
            y1 = int((y_center - height / 2) * h)
            x2 = int((x_center + width / 2) * w)
            y2 = int((y_center + height / 2) * h)

            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                img,
                class_names[class_id],
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.9,
                (0, 255, 0),
                2,
            )

    return img


train_img_dir = Path(dataset.location) / "train" / "images"
train_label_dir = Path(dataset.location) / "train" / "labels"

sample_images = list(train_img_dir.glob("*.jpg"))[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.ravel()

for idx, img_path in enumerate(sample_images):
    label_path = train_label_dir / (img_path.stem + ".txt")
    img = visualize_sample_with_labels(img_path, label_path, data_config["names"])
    axes[idx].imshow(img)
    axes[idx].set_title(f"Sample {idx + 1}")
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from ultralytics import YOLO

if fm.is_colab:
    fm.mount_drive()

# Colab環境の場合、一時ディレクトリに確実に保存するためカレントディレクトリを変更
if fm.is_colab:
    os.chdir("/content")  # 揮発性の/contentに移動
    print(f"✓ カレントディレクトリを変更: {os.getcwd()}")

# モデルサイズの選択: n(nano), s(small), m(medium), l(large), x(xlarge)
model = YOLO(
    "/content/drive/MyDrive/Visuable_for_you_tabletennis/models/pretrained/yolo11s.pt"
)

results = model.train(
    data=data_yaml_path,  # データセット設定ファイル
    epochs=100,  # エポック数
    imgsz=640,  # 入力画像サイズ
    batch=16,  # バッチサイズ（GPU メモリに応じて調整）
    name="table_detector_v1",  # プロジェクト名
    patience=20,  # Early stoppingのpatience
    save=True,  # モデルの保存
    save_period=10,  # 10エポックごとにチェックポイント保存
    cache=True,  # データセットをキャッシュして高速化
    device=0,  # GPU使用（CPU の場合は 'cpu'）
    workers=2,  # データローダーのワーカー数
    project="/content/table_detection",  # 一時ディレクトリに保存（/contentは揮発性）
    exist_ok=True,  # 既存のプロジェクトを上書き
    pretrained=True,  # 事前学習済み重みを使用
    optimizer="auto",  # オプティマイザー自動選択
    verbose=True,  # 詳細なログ出力
    seed=42,  # 再現性のためのシード値
    deterministic=True,  # 決定的な訓練
    single_cls=False,  # マルチクラス検出
    rect=False,  # 矩形訓練
    cos_lr=True,  # コサインLRスケジューラ
    close_mosaic=10,  # 最後の10エポックでモザイク拡張を無効化
    amp=True,  # 自動混合精度訓練
    fraction=1.0,  # 使用するデータセットの割合
    profile=False,  # プロファイリング
    freeze=None,  # 凍結するレイヤー数
    lr0=0.01,  # 初期学習率
    lrf=0.01,  # 最終学習率係数
    momentum=0.937,  # モメンタム
    weight_decay=0.0005,  # 重み減衰
    warmup_epochs=3.0,  # ウォームアップエポック数
    warmup_momentum=0.8,  # ウォームアップモメンタム
    warmup_bias_lr=0.1,  # ウォームアップバイアス学習率
    box=7.5,  # ボックスロスの重み
    cls=0.5,  # クラスロスの重み
    dfl=1.5,  # DFLロスの重み
    pose=12.0,  # ポーズロスの重み（検出タスクでは無視される）
    kobj=1.0,  # キーポイント objロスの重み
    label_smoothing=0.0,  # ラベルスムージング
    nbs=64,  # 正規化バッチサイズ
    hsv_h=0.015,  # HSV色相拡張
    hsv_s=0.7,  # HSV彩度拡張
    hsv_v=0.4,  # HSV明度拡張
    degrees=0.0,  # 回転拡張（度）
    translate=0.1,  # 平行移動拡張
    scale=0.5,  # スケール拡張
    shear=0.0,  # せん断拡張
    perspective=0.0,  # 透視変換拡張
    flipud=0.0,  # 上下反転確率
    fliplr=0.5,  # 左右反転確率
    mosaic=1.0,  # モザイク拡張確率
    mixup=0.0,  # ミックスアップ拡張確率
    copy_paste=0.0,  # コピー&ペースト拡張確率
)

# 出力先を確認
print(f"\n✓ 訓練が完了しました！")
print(f"✓ 出力ディレクトリ: /content/table_detection/table_detector_v1/")
print(f"  （注意: /contentは一時ディレクトリ。セッション終了で削除されます）")
print(f"\n次のセルで評価を実行し、セル10でGoogle Driveに永久保存します。")

# /content/table_detection/table_detector_v1/
# ├── results.png                  # 訓練の損失・メトリクスグラフ
# ├── confusion_matrix.png         # 混同行列
# ├── val_batch0_pred.jpg          # バリデーション予測結果
# ├── val_batch1_pred.jpg
# ├── val_batch2_pred.jpg
# ├── train_batch*.jpg             # 訓練バッチの可視化
# └── weights/
#     ├── best.pt                  # ベストモデル
#     ├── last.pt                  # 最終エポックのモデル
#     └── epoch*.pt                # 10エポックごとのチェックポイント

In [ ]:
metrics = model.val()

print("\n=== 評価結果 ===")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
from IPython.display import Image, display
import glob

results_dir = "/content/table_detection/table_detector_v1"

results_plot = os.path.join(results_dir, "results.png")
if os.path.exists(results_plot):
    print("訓練結果:")
    display(Image(filename=results_plot))

confusion_matrix = os.path.join(results_dir, "confusion_matrix.png")
if os.path.exists(confusion_matrix):
    print("\n混同行列:")
    display(Image(filename=confusion_matrix))

val_batches = sorted(glob.glob(os.path.join(results_dir, "val_batch*_pred.jpg")))
if val_batches:
    print("\nバリデーション予測結果:")
    for batch_img in val_batches[:2]:
        display(Image(filename=batch_img))

In [ ]:
best_model_path = os.path.join(results_dir, "weights", "best.pt")
best_model = YOLO(best_model_path)

print(f"ベストモデル: {best_model_path}")

onnx_path = best_model.export(format="onnx")
print(f"\nONNXモデルをエクスポートしました: {onnx_path}")

torchscript_path = best_model.export(format="torchscript")
print(f"TorchScriptモデルをエクスポートしました: {torchscript_path}")

In [ ]:
from utils import ModelFileManager

if fm.is_colab:
    fm.mount_drive()

mfm = ModelFileManager(fm)

if fm.is_colab:
    drive_model_dir = (
        "/content/drive/MyDrive/Visuable_for_you_tabletennis/models/table_detection"
    )
else:
    drive_model_dir = str(fm.get_path("models/table_detection"))

files_to_copy = ["best.pt", "best.onnx", "best.torchscript"]
weights_dir = os.path.join(results_dir, "weights")

success_count, actual_drive_path = mfm.save_to_drive(
    source_dir=weights_dir,
    drive_path=drive_model_dir,
    files_to_copy=files_to_copy,
    use_timestamp=True,
)

print(f"\n✓ モデルをGoogle Driveに保存しました: {actual_drive_path}")

In [ ]:
valid_img_dir = Path(dataset.location) / "valid" / "images"
test_images = list(valid_img_dir.glob("*.jpg"))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

for idx, img_path in enumerate(test_images):
    results = best_model.predict(
        source=str(img_path),
        conf=0.25,  # 信頼度閾値
        iou=0.45,  # NMS IOU閾値
        verbose=False,
    )

    result_img = results[0].plot()
    result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)

    axes[idx].imshow(result_img)
    axes[idx].set_title(f"Prediction {idx + 1}")
    axes[idx].axis("off")

plt.tight_layout()
plt.show()

print("\n最初のテスト画像の検出結果:")
first_result = best_model.predict(source=str(test_images[0]), verbose=False)[0]
for box in first_result.boxes:
    class_id = int(box.cls[0])
    confidence = float(box.conf[0])
    bbox = box.xyxy[0].tolist()
    print(
        f"クラス: {data_config['names'][class_id]}, 信頼度: {confidence:.2f}, BBox: {bbox}"
    )

In [ ]:
print("=== モデル情報 ===")
print(f"モデルアーキテクチャ: YOLOv11n")
print(f"入力サイズ: 640x640")
print(f"クラス数: {len(data_config['names'])}")
print(f"クラス名: {data_config['names']}")
print(f"\n訓練データ数: {train_count}")
print(f"検証データ数: {valid_count}")
print(f"\n最終評価指標:")
print(f"  - mAP50: {metrics.box.map50:.4f}")
print(f"  - mAP50-95: {metrics.box.map:.4f}")
print(f"  - Precision: {metrics.box.mp:.4f}")
print(f"  - Recall: {metrics.box.mr:.4f}")
print(f"\nモデル保存先:")
print(f"  - PyTorch: {best_model_path}")
print(f"  - ONNX: {onnx_path}")
print(f"  - TorchScript: {torchscript_path}")
print(f"  - Google Drive: {actual_drive_path}")

if fm.is_colab:
    print("\n結果ファイルをダウンロードしますか？")
    print("以下のコメントを外して実行してください：")
    print("# fm.download_files([best_model_path, onnx_path, torchscript_path])")